# Chapter 9 §9.7: Financial Analyst Agent

ProgramOfThought handles calculation, while RLM explores long filings and a trajectory metric penalizes unsupported intermediate values and drift.


In [ ]:
%pip install -r ../requirements.txt -q


## Runtime setup

The cells tagged `chapter-example` reproduce the manuscript code blocks verbatim. Supporting cells only provide imports, fixtures, or safe local stand-ins for external services.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

import dspy

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY in ../.env before running this notebook.")

lm = dspy.LM("openai/gpt-4o-mini", api_key=api_key)
dspy.configure(lm=lm)


## ProgramOfThought calculation


In [ ]:
class FinancialAnalysis(dspy.Signature):
    """Analyze financial data to answer a question."""

    ticker = dspy.InputField(desc="stock ticker symbol")
    question = dspy.InputField(desc="financial question")
    stock_data = dspy.InputField(desc="current stock price data")
    financials = dspy.InputField(desc="company financial metrics")

    answer = dspy.OutputField(desc="the answer to the question")

analyzer = dspy.ProgramOfThought(FinancialAnalysis)


## RLM over long financial documents


In [ ]:
financial_documents = """Example Corp 2025 revenue: $100 million.
Example Corp 2026 revenue: $112 million.
Source: audited annual reports for fiscal years 2025 and 2026."""


In [ ]:
rlm = dspy.RLM(
    "documents, question -> answer",
    max_iterations=10,
    max_llm_calls=50,
    sub_lm=dspy.LM("openai/gpt-5.6-luna")
)

result = rlm(
    documents=financial_documents,
    question="What was the year-over-year revenue growth rate?"
)


## Trajectory-aware evaluation


In [ ]:
import math
import re

class TrajectoryJudge(dspy.Signature):
    """Evaluate whether an agent's reasoning trajectory shows signs
    of Context Poisoning (intermediate values not supported by the
    source documents) or Context Drift (wandering into tangential
    queries unrelated to the original question)."""

    question = dspy.InputField(desc="the original question")
    documents = dspy.InputField(desc="the source documents")
    trajectory = dspy.InputField(
        desc="the agent's step-by-step reasoning trace"
    )

    poisoning_detected: bool = dspy.OutputField(
        desc="True if intermediate values are unsupported by sources"
    )
    drift_detected: bool = dspy.OutputField(
        desc="True if the agent explored tangential topics"
    )
    reasoning = dspy.OutputField(desc="brief reason for the judgment")


trajectory_judge = dspy.ChainOfThought(TrajectoryJudge)

FINANCIAL_NUMBER = re.compile(
    r"(?P<sign>-)?\s*[$€£]?\s*"
    r"(?P<number>\d[\d,]*(?:\.\d+)?)\s*"
    r"(?P<suffix>%|[KMB]|thousand|million|billion)?",
    re.IGNORECASE,
)

MULTIPLIERS = {
    "": 1.0,
    "K": 1_000.0,
    "THOUSAND": 1_000.0,
    "M": 1_000_000.0,
    "MILLION": 1_000_000.0,
    "B": 1_000_000_000.0,
    "BILLION": 1_000_000_000.0,
    "%": 0.01,
}

def parse_financial_number(value):
    """Normalize a formatted financial value to a float."""
    match = FINANCIAL_NUMBER.search(str(value))
    if not match:
        raise ValueError(f"No financial number found in {value!r}")

    number = float(match.group("number").replace(",", ""))
    suffix = (match.group("suffix") or "").upper()
    sign = -1.0 if match.group("sign") else 1.0

    return sign * number * MULTIPLIERS[suffix]

def trajectory_metric(example, pred, trace=None):
    # Check final answer correctness first (cheap, deterministic)

    try:
        predicted_value = parse_financial_number(pred.answer)
        expected_value = parse_financial_number(example.answer)
        answer_correct = math.isclose(
            predicted_value,
            expected_value,
            rel_tol=0.05,
            abs_tol=1e-9,
        )
    except (TypeError, ValueError):
        answer_correct = False

    if not answer_correct:
        return 0.0

    # Use an LLM judge to inspect the trajectory itself
    judgment = trajectory_judge(
        question=example.question,
        documents=example.documents,
        trajectory=str(pred.trajectory)
    )

    if judgment.poisoning_detected:
        return 0.3  # Correct answer, suspicious path
    if judgment.drift_detected:
        return 0.7  # Correct answer, wandering path
    return 1.0


## Deterministic parser smoke checks


In [ ]:
assert parse_financial_number("$1.25B") == 1_250_000_000.0
assert math.isclose(parse_financial_number("-2.8%"), -0.028)
assert parse_financial_number("€42 million") == 42_000_000.0
print("Financial-number parser smoke checks passed.")
